# Load Dimension Tables From CSV

This notebook loads some dimension tables directly from CSV files which are
already curated and cleaned. No need for a bronze load process for this data.

Starts with `%run "../../libs/notebook_init"` — see
`.claude/project/helpers.md` for what `notebook_init` injects (CATALOG,
BRONZE, SILVER, GOLD, AUDIT, RAW_FILES, STATUS_*, PIPELINE_RUN_ID, Utils, F,
datetime, etc.).


In [0]:
%run "../../libs/notebook_init"

In [ ]:
# Imports and constants specific to the dim load from CSV.
# SOURCE_SUBPATH is the masterdata subfolder under RAW_FILES that holds the
# five pre-curated CSVs (Currency, Dates, ExchangeRates, Store, Territory).
# Each per-dim cell below sets its own TARGET_TABLE.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone

# transform_detail_log_insert isn't in notebook_init's central import yet —
# pull it in here so this notebook can log per-dim audit rows.
from pipeline_logging import transform_detail_log_insert

SOURCE_SUBPATH = "masterdata"                       # case must match volume
SOURCE_PATH    = f"{RAW_FILES}{SOURCE_SUBPATH}"


In [ ]:
nb = Utils.get_notebook_context(dbutils)
notebook_folder = nb['notebook_folder']
notebook_name = nb['notebook_name']

step_log_id       = str(uuid.uuid4())
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
layer             = "silver"
target_table      = None
status            = STATUS_RUNNING
started_timestamp = datetime.now(timezone.utc)
rows_read         = 0
rows_written      = 0
error_message     = None


pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table)

In [ ]:
TARGET_TABLE   = f"{SILVER}.dim_currency"

# Define the SQL logic for this specific dimension
currency_merge_sql = lambda t: f"""
    MERGE INTO {t} a
    USING temp_dim t
    ON a.CurrencyCode = t.CurrencyCode
    WHEN NOT MATCHED THEN
    INSERT (CurrencyCode, CurrencyName, InsertedDate, UpdatedDate)
    VALUES (t.CurrencyCode, t.CurrencyName, t.InsertedDate, t.UpdatedDate)
"""

try:
    result = Utils.load_dim_from_csv(
        spark=spark,
        source_path=f"{SOURCE_PATH}/Currency.csv",
        target_table=TARGET_TABLE,
        merge_sql_fn=currency_merge_sql,
        add_timestamps=True,
    )

    transform_detail_log_insert(
        spark,
        pipeline_run_id=PIPELINE_RUN_ID,
        step_log_id=step_log_id,
        **result,
    )

    # load_dim_from_csv catches its own exceptions and returns
    # {"status": "failed", ...} rather than raising. Surface that here so
    # the notebook step_log row flips to STATUS_FAILED instead of staying
    # as STATUS_RUNNING.
    if result.get("status") == STATUS_FAILED:
        raise RuntimeError(
            f"load_dim_from_csv failed for {TARGET_TABLE}: "
            f"{result.get('error_message', 'no error_message returned')}"
        )

    # Accumulate per-step counts for the step-log close-out at notebook end.
    rows_read    += result.get("rows_read", 0) or 0
    rows_written += result.get("rows_written", 0) or 0

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    # NOTE: this is a multi-transform notebook. By the time we hit this
    # except path on dim N, dims 1..N-1 have already committed real rows
    # to their target tables. The accumulated rows_read / rows_written
    # values are the most honest summary at the step-log level — they
    # represent the actual work this step performed before failure.
    # Setting them to 0 would erase real persisted data from the audit
    # trail. Per-dim granular detail is captured in transform_detail_log.
    # Single-transform sibling notebooks (slvr_02, bronze) pass 0 because
    # a single failed MERGE means nothing committed; that premise does
    # not apply here.
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )
    raise


In [ ]:
TARGET_TABLE   = f"{SILVER}.dim_date"

# Define the SQL logic for this specific dimension
date_merge_sql = lambda t: f"""
  MERGE INTO {TARGET_TABLE} AS target
USING temp_dim AS source
ON target.YearMonth = source.YearMonth
WHEN MATCHED THEN
  UPDATE SET
    target.DateYear = source.DateYear,
    target.DateMonth = source.DateMonth,
    target.LastDayOfMonth = to_date(source.LastDayOfMonth, 'M/d/yyyy'),
    target.Quarter = source.Quarter,
    target.Season = source.Season
WHEN NOT MATCHED THEN
  INSERT (DateYear, DateMonth, YearMonth, LastDayOfMonth, Quarter, Season)
  VALUES (source.DateYear, source.DateMonth, source.YearMonth, to_date(source.LastDayOfMonth, 'M/d/yyyy'), source.Quarter, source.Season
  )
"""

try:
    result = Utils.load_dim_from_csv(
        spark=spark,
        source_path=f"{SOURCE_PATH}/Dates.csv",
        target_table=TARGET_TABLE,
        merge_sql_fn=date_merge_sql,
        add_timestamps=True,
    )

    transform_detail_log_insert(
        spark,
        pipeline_run_id=PIPELINE_RUN_ID,
        step_log_id=step_log_id,
        **result,
    )

    if result.get("status") == STATUS_FAILED:
        raise RuntimeError(
            f"load_dim_from_csv failed for {TARGET_TABLE}: "
            f"{result.get('error_message', 'no error_message returned')}"
        )

    rows_read    += result.get("rows_read", 0) or 0
    rows_written += result.get("rows_written", 0) or 0

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    # Cumulative rows_read / rows_written on failure — see cell 4 NOTE
    # for the multi-transform rationale.
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )
    raise


In [ ]:
TARGET_TABLE   = f"{SILVER}.dim_exchange_rate"

# Define the SQL logic for this specific dimension
er_merge_sql = lambda t: f"""
    MERGE INTO {TARGET_TABLE} a
            USING temp_dim t
            ON a.FromCurrency = t.FromCurrency
            AND a.ToCurrency = t.ToCurrency
            AND a.EffectiveDate = t.EffectiveDate
            WHEN MATCHED AND a.ExchangeRate <> t.AverageRate THEN
                UPDATE SET
                a.ExchangeRate = t.AverageRate,
                a.UpdatedDate = t.UpdatedDate
            WHEN NOT MATCHED THEN
                INSERT (FromCurrency, ToCurrency, EffectiveDate, ExchangeRate, InsertedDate, UpdatedDate  )
                 VALUES ( t.FromCurrency, t.ToCurrency, t.EffectiveDate,  t.AverageRate,  t.InsertedDate,  t.UpdatedDate )
            """
df_transform = lambda df: df.withColumn(
                            "EffectiveDate", F.to_timestamp("EffectiveDate", 'M/d/yyyy'))

try:
    result = Utils.load_dim_from_csv(
        spark=spark,
        source_path=f"{SOURCE_PATH}/ExchangeRates.csv",
        target_table=TARGET_TABLE,
        merge_sql_fn=er_merge_sql,
        add_timestamps=True,
        df_transform=df_transform,
    )

    transform_detail_log_insert(
        spark,
        pipeline_run_id=PIPELINE_RUN_ID,
        step_log_id=step_log_id,
        **result,
    )

    if result.get("status") == STATUS_FAILED:
        raise RuntimeError(
            f"load_dim_from_csv failed for {TARGET_TABLE}: "
            f"{result.get('error_message', 'no error_message returned')}"
        )

    rows_read    += result.get("rows_read", 0) or 0
    rows_written += result.get("rows_written", 0) or 0

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    # Cumulative rows_read / rows_written on failure — see cell 4 NOTE.
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )
    raise


In [ ]:
TARGET_TABLE   = f"{SILVER}.dim_store"

# Define the SQL logic for this specific dimension
store_merge_sql = lambda t: f"""
     MERGE INTO {TARGET_TABLE} a
            USING temp_dim t
            ON a.StoreName = t.StoreName
            WHEN MATCHED AND (a.StoreType <> t.StoreType OR a.Description <> t.Description) THEN
                UPDATE SET
                a.StoreType = t.StoreType,
                a.Description = t.Description,
                a.UpdatedDate = t.UpdatedDate
            WHEN NOT MATCHED THEN
                INSERT (StoreName, StoreType, Description, InsertedDate, UpdatedDate)
                VALUES (t.StoreName, t.StoreType, t.Description, t.InsertedDate, t.UpdatedDate)
"""

try:
    result = Utils.load_dim_from_csv(
        spark=spark,
        source_path=f"{SOURCE_PATH}/Store.csv",
        target_table=TARGET_TABLE,
        merge_sql_fn=store_merge_sql,
        add_timestamps=True,
    )

    transform_detail_log_insert(
        spark,
        pipeline_run_id=PIPELINE_RUN_ID,
        step_log_id=step_log_id,
        **result,
    )

    if result.get("status") == STATUS_FAILED:
        raise RuntimeError(
            f"load_dim_from_csv failed for {TARGET_TABLE}: "
            f"{result.get('error_message', 'no error_message returned')}"
        )

    rows_read    += result.get("rows_read", 0) or 0
    rows_written += result.get("rows_written", 0) or 0

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    # Cumulative rows_read / rows_written on failure — see cell 4 NOTE.
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )
    raise


In [ ]:
TARGET_TABLE   = f"{SILVER}.dim_territory"

# Define the SQL logic for this specific dimension
store_merge_sql = lambda t: f"""
   MERGE INTO {TARGET_TABLE} AS target
  USING temp_dim AS source
  ON target.TerritoryCode = source.TerritoryCode
  WHEN MATCHED THEN
    UPDATE SET
      target.TerritoryName = source.TerritoryName,
      target.TradeRegion   = source.TradeRegion,
      target.Continent     = source.Continent,
      target.UpdatedDate   = source.UpdatedDate
  WHEN NOT MATCHED THEN
      INSERT (TerritoryCode, TerritoryName, TradeRegion, Continent, InsertedDate, UpdatedDate)
      VALUES (source.TerritoryCode, source.TerritoryName, source.TradeRegion, source.Continent, source.InsertedDate, source.UpdatedDate)
"""

df_transform = lambda df: (
                            df.withColumn("InsertedDate", F.current_timestamp( ) )
                              .withColumn("UpdatedDate", F.current_timestamp())
                        )

try:
    result = Utils.load_dim_from_csv(
        spark=spark,
        source_path=f"{SOURCE_PATH}/Territory.csv",
        target_table=TARGET_TABLE,
        merge_sql_fn=store_merge_sql,
        add_timestamps=True,
        df_transform=df_transform,
    )

    transform_detail_log_insert(
        spark,
        pipeline_run_id=PIPELINE_RUN_ID,
        step_log_id=step_log_id,
        **result,
    )

    if result.get("status") == STATUS_FAILED:
        raise RuntimeError(
            f"load_dim_from_csv failed for {TARGET_TABLE}: "
            f"{result.get('error_message', 'no error_message returned')}"
        )

    rows_read    += result.get("rows_read", 0) or 0
    rows_written += result.get("rows_written", 0) or 0

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    # Cumulative rows_read / rows_written on failure — see cell 4 NOTE.
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )
    raise


In [ ]:
# Notebook-level step-log close-out. Each per-dim cell above either
# accumulated rows_read / rows_written on success or raised + flipped this
# step to STATUS_FAILED on the except path. Reaching this cell means every
# per-dim cell completed without raising.

ended_timestamp = datetime.now(timezone.utc)
status = STATUS_SUCCEEDED

pipeline_step_log_upsert(
    spark, step_log_id, pipeline_run_id, step_sequence,
    notebook_folder, notebook_name, status, started_timestamp,
    layer, target_table, rows_read, rows_written, ended_timestamp
)

print(f"Success: slvr_01 step closed. rows_read={rows_read}, rows_written={rows_written}")


In [0]:
%skip
%sql
SELECT 'dim_currency row count = '      || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_currency
UNION ALL
SELECT 'dim_date row count = '          || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_date
UNION ALL
SELECT 'dim_exchange_rate row count = ' || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_exchange_rate
UNION ALL
SELECT 'dim_store row count = '         || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_store
UNION ALL
SELECT 'dim_territory row count = '     || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_territory;